## Test Day

In [9]:
if False:
    import os
    os.environ["QUANTDATA_CACHE_MODE"] = "off" # turns off global cache mode; good for testing, but do not use in general.

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree

from datetime import date
from quantdata.widb.day import collect_raw_day, enrich_day, ExpiryFilter, RawDayFilter
from quantdata.widb.arbfree import mk_arb_free_day, MkArbFree
from quantdata.biz_calendar import next_business_day
from cdxcore.verbose import Context
from IPython.display import display

def test():
    ticker = "SPX"
    d      = next_business_day( date(2006, 1, 19), count_start=True )

    raw_day_filter = RawDayFilter()
    expiry_filter = ExpiryFilter()
    verbose = Context.all

    raw_day = collect_raw_day(ticker, d, raw_day_filter=raw_day_filter, verbose=verbose, raise_if_bad=False)
    #display(raw_day)

    enr_day = enrich_day(ticker, d, raw_day, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad=False, override_cache_mode="off")
    #display(enr_day)

    err = enr_day[enr_day['pure_ask_price'] <= enr_day['pure_bid_price']]
    if len(err) > 0:
        display(err[['expiry','strike','pure_ask_price','pure_bid_price']])
        raise ValueError("Bad data: found options with ask price below or equal to bid price.")
    
    arb_free = MkArbFree()
    arb_free = mk_arb_free_day(ticker, d, enr_day, mk_arb_free=arb_free, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad=False, override_cache_mode="off")

    display(arb_free)
    return arb_free

_ = test()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[ 0.46900921  0.43124276  0.39363494  0.35622541  0.31893486  0.30066633
  0.2823978   0.2642879   0.24981585  0.24586074  0.24261948  0.22838537
  0.21075137  0.19351394  0.19011405  0.18663483  0.17643515  0.15975293
  0.14354661  0.12771751  0.11240302  0.10946729  0.10647569  0.09778301
  0.08401536  0.07871189  0.07609981  0.07099463  0.05896535  0.05223953
  0.04800682  0.03823803  0.02958473  0.02236701  0.01649765  0.01173871
  0.0101524   0.00951788  0.00809019  0.00539346  0.00237947  0.00099145
 -0.03799196]
[ 0.46881092  0.43084618  0.39335734  0.35582884  0.31857794  0.30011112
  0.28184259  0.26357406  0.24910201  0.24554348  0.24190564  0.22767153
  0.21003753  0.19256215  0.18916226  0.18568305  0.17548336  0.15880114
  0.14235688  0.12652778  0.11121328  0.10827756  0.10528596  0.09659327
  0.08258768  0.07728421  0.07467213  0.06956695  0.05753766  0.05081185
  0.04657914  0.036810

/home/hans/vqdata/lib/python3.12/site-packages/cdxcore/subdir.py:135: UserWarning: SPX 2006-01-19: found synthetic call bid price or spread below zero for expiry_id 0.20 business days to expiry -- this indicates data issues we cannot resolve. Removing those strikes
/home/hans/vqdata/lib/python3.12/site-packages/cdxcore/subdir.py:135: UserWarning: SPX 2006-01-19: found synthetic call bid price or spread below zero for expiry_id 20.20 business days to expiry -- this indicates data issues we cannot resolve. Removing those strikes
/home/hans/vqdata/lib/python3.12/site-packages/cdxcore/subdir.py:135: UserWarning: SPX 2006-01-19: found synthetic call bid price or spread below zero for expiry_id 39.20 business days to expiry -- this indicates data issues we cannot resolve. Removing those strikes
/home/hans/vqdata/lib/python3.12/site-packages/cdxcore/subdir.py:135: UserWarning: SPX 2006-01-19: found synthetic call bid price or spread below zero for expiry_id 102.20 business days to expiry -- t

01:   SPX 2006-01-19: Generated 5 arbitrage-free expiries with extended strikes from 5 market expiries. This took 0.21s.
01:   SPX 2006-01-19: Finding arbitrage-free mid prices accross the entire surface of 131 strikes with a minimum strike distance of 0.00388... done. This took 0.16s.
00: SPX 2006-01-19: generated arbitrage-free price surface. Dropped 0 expiries and 43 strikes of the remaining expiries. This took 0.45s.


,spot,spot_prev,date,ticker,call_symbol,put_symbol,call_opt_id,put_opt_id,is_american,is_am,...,pure_fit1_delta,pure_fit1_dk,pure_fit1_gamma,pure_fit1_vega,pure_fit1_theta,pure_fitX_delta,pure_fitX_dk,pure_fitX_gamma,pure_fitX_vega,pure_fitX_theta
0,1285.040039,1277.930054,2006-01-19,SPX,SZP.AG,SZP.MG,31722719,31722718,False,True,...,0.971388,-0.970034,3.220198,0.001843,0.837201,0.972382,-0.971077,3.152434,0.001789,0.806319
1,1285.040039,1277.930054,2006-01-19,SPX,SZP.AJ,SZP.MJ,31708833,31708836,False,True,...,0.959366,-0.958024,5.749376,0.002459,0.834756,0.960262,-0.958951,5.680059,0.002415,0.814900
2,1285.040039,1277.930054,2006-01-19,SPX,SZP.AK,SZP.MK,31731886,31731888,False,True,...,0.953350,-0.952030,7.294329,0.002749,0.822219,0.954177,-0.952883,7.227162,0.002710,0.806356
3,1285.040039,1277.930054,2006-01-19,SPX,SZP.AL,SZP.ML,31731887,31731885,False,True,...,0.945419,-0.944134,9.601152,0.003115,0.802240,0.946120,-0.944853,9.541484,0.003083,0.790900
4,1285.040039,1277.930054,2006-01-19,SPX,SZP.AM,SZP.MM,31726956,31726957,False,True,...,0.934289,-0.933053,13.332010,0.003602,0.772639,0.934813,-0.933589,13.285795,0.003580,0.765753
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,1285.040039,1277.930054,2006-01-19,SPX,SXZ.LI,SXZ.XI,31787278,31787279,False,True,...,0.177911,-0.151753,2.466907,0.248416,0.015120,0.177854,-0.151706,2.466933,0.248366,0.015114
160,1285.040039,1277.930054,2006-01-19,SPX,SXZ.LM,SXZ.XM,31766939,31766938,False,True,...,0.168723,-0.143559,2.392853,0.240170,0.014570,0.168678,-0.143522,2.392845,0.240129,0.014565
161,1285.040039,1277.930054,2006-01-19,SPX,SXZ.LO,SXZ.XO,31809728,31809727,False,True,...,0.150105,-0.127121,2.240264,0.222465,0.013353,0.150087,-0.127106,2.240236,0.222447,0.013351
162,1285.040039,1277.930054,2006-01-19,SPX,SXM.LT,SXM.XT,23948257,23948256,False,True,...,0.109085,-0.091286,1.842022,0.178255,0.010426,0.109085,-0.091286,1.842022,0.178255,0.010426
